**Real Estate SMS Marketing - ETL Pipeline**

The goal of this project is to use python and pandas to make a full ETL process preparing the data for a later use in sms messages.

With a dataset provided, we need to get the full address for each property, leave only one mobile phone number, and create 3 different sms messages that iterates on all the properties. The final result should be an Excel file with 3 columns, Full Address, Mobile, and SMS.


For this project it will be necessary to use pandas and openpyxl (previously installed)

In [ ]:
import pandas as pd

df = pd.read_excel('Property Export expired raw.xlsx')
df.info()

1. Convert the ZIP to str for better concat for a full address

In [ ]:
df['Zip'] =df['Zip'].astype(str)

#Create copy for cleaned df

df_cleaned = df.copy()

2. Concatenate the columns for full address

In [ ]:
df_cleaned['Full Address'] = df_cleaned[['Address', 'City', 'State', 'Zip']].agg(', '.join, axis=1)

3. Clean the dataset from the mobile column as it has Non values, and for this intended purpose, we will take only those that have a mobile phone number, the other can be dropped.

In [ ]:
df_cleaned.dropna(subset=['Mobile'], inplace=True)

#Check if NaN values have been dropped
df_cleaned['Mobile'].isna().sum()

4. Clean the mobile column, as some rows have multiple phone numbers, we will use only the first one

In [ ]:
df_cleaned['Mobile Cleaned'] = df_cleaned['Mobile'].apply(lambda x: x.split(',')[0].strip())

5. As the other columns are not needed we will keep only those cleaned

In [ ]:
df_cleaned = df_cleaned[['Full Address', 'Mobile Cleaned']]

6. Now we have 3 sms messages that we have to shift so not all the sms are equal, and they should go with their respective address

In [ ]:
# First, I'll need to reset the index for a better approach
df_cleaned.reset_index(drop=True, inplace=True)
df_cleaned.index

In [ ]:
# Create the messages:
message1 = "Hi, quick question about your home on {property_address}. If the right opportunity presented itself, would you consider selling?"
message2 = "Hi! Reaching out about your place on {property_address}. Would you be open to discussing a possible sale if the numbers made sense?"
message3 = "Hi, I’m inquiring about the home on {property_address}. Would you consider an offer if it was a good fit for you?"

messages = [message1, message2, message3]


In order to avoid the sms be taken as spam we will use modulo % 3 to iterate over the 3 messages index to get every 3 messages a "different one" using the property address for each one.

In [ ]:
#Apply the message to each address

df_cleaned['SMS'] = df_cleaned.apply(lambda row: messages[row.name%3].format(property_address = row['Full Address']), axis=1)

In [ ]:
df_cleaned

7. Export the cleaned df for its use

In [ ]:
df_cleaned.to_excel('Properties_for_SMS.xlsx')